# Trees and Model Stability

Trees are notorious for being **unstable**: Small changes in the data can lead to noticeable or large changes in the tree. We're going to explore this phenomenon.

1. Check out the data set `heart_failure` and appropriately clean it.
2. Perform a train-test split for a specific seed (save the seed for reproducibility). Fit a tree and a linear model on the training data and evaluate their performance on the test data. Set aside the predictions these models make.
3. Repeat step 2 for three to five different seeds (save the seeds for reproducibility). How different are the trees that you get? Your linear model coefficients?. Set aside the predictions these models make.
4. Typically, you would see the trees changing what appears to be a non-trivial amount, while the linear model coefficients don't vary nearly as much. Often, the changes appear substantial. 
But are they? Instead of focusing on the tree or model coefficients, do three things:
    
    a. Make scatterplots of the predicted values on the test set from question 2 against the predicted values for the alternative models from part 3, separately for your trees and linear models. Do they appear reasonably similar?
    
    b. Compute the correlation between your model in part 2 and your alternative models in part 3, separately for your trees and linear models. Are they highly correlated or not?
    
    c. Run a simple linear regression of the predicted values on the test set from the alternative models on the predicted values from question 2, separately for your trees and linear models. Is the intercept close to zero? Is the slope close to 1? Is the $R^2$ close to 1?

5. Do linear models appear to have similar coefficients and predictions across train/test splits? Do trees?
6. True or false for this dataset, and explain: "Even if the models end up having a substantially different appearance, the predictions they generate are often very similar."

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Ameilia Chen, Lawson Pham, Nate Park, Amran Belete

**Date:** 11/16/2026

1. Check out the data set `heart_failure` and appropriately clean it.

In [4]:
# Your Phase 1 solution to the problem goes here.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df = pd.read_csv('data/heart_failure/heart_failure_clinical_records_dataset.csv', low_memory=False)
print("df.shape:")
print(df.shape, '\n')

print("df.dtypes:")
print(df.dtypes, '\n')


print("Original shape:", df.shape)
print("Exact duplicates:", df.duplicated().sum())
print("Missing values:", df.isna().sum().sum())
print("Outcome counts:")
display(df["DEATH_EVENT"].value_counts().sort_index())

df = df.drop_duplicates().copy()
df = df.apply(pd.to_numeric, errors="raise")
assert df.isna().sum().sum() == 0
assert df["DEATH_EVENT"].isin([0, 1]).all()



df.head()

df.shape:
(299, 13) 

df.dtypes:
age                         float64
anaemia                       int64
creatinine_phosphokinase      int64
diabetes                      int64
ejection_fraction             int64
high_blood_pressure           int64
platelets                   float64
serum_creatinine            float64
serum_sodium                  int64
sex                           int64
smoking                       int64
time                          int64
DEATH_EVENT                   int64
dtype: object 

Original shape: (299, 13)
Exact duplicates: 0
Missing values: 0
Outcome counts:


DEATH_EVENT
0    203
1     96
Name: count, dtype: int64

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


2. Perform a train-test split for a specific seed (save the seed for reproducibility). Fit a tree and a linear model on the training data and evaluate their performance on the test data. Set aside the predictions these models make.
3. Repeat step 2 for three to five different seeds (save the seeds for reproducibility). How different are the trees that you get? Your linear model coefficients?. Set aside the predictions these models make.



In [12]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


X = df.drop(columns=["DEATH_EVENT"])
y = df["DEATH_EVENT"]

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=90
)

seeds = [42, 65, 28, 2]
tree_models = {}
linear_models = {}
metrics = []
coefficients = []
tree_summary = []


prediction = pd.DataFrame({"actual": y_test}, index=y_test.index)
for seed in seeds:
    X_train, _, y_train, _ = train_test_split(
        X_dev, y_dev, test_size=0.20, random_state=seed
    )

    tree = DecisionTreeClassifier(random_state=seed)
    linear = LogisticRegression(random_state=seed, max_iter=1000)

    tree.fit(X_train, y_train)
    linear.fit(X_train, y_train)

    tree_models[seed] = tree
    linear_models[seed] = linear

    for model_name, model in [("tree", tree), ("linear", linear)]:
        predict = model.predict(X_test)
        prediction[f"{model_name}_{seed}"] = predict

        metrics.append({
            "seed": seed,
            "model": model_name,
            "MAE": mean_absolute_error(y_test, predict),
            "RMSE": np.sqrt(mean_squared_error(y_test, predict)),
            "R2": r2_score(y_test, predict)
        })

    coefficients.append({
        "seed": seed,
        "intercept": linear.intercept_[0],
    })

    root_feature = X.columns[tree.tree_.feature[0]]
    tree_summary.append({
        "seed": seed,
        "depth": tree.get_depth(),
        "leaves": tree.get_n_leaves(),
        "root_feature": root_feature,
        "root_threshold": tree.tree_.threshold[0],
    })


metrics = pd.DataFrame(metrics)
coefficients = pd.DataFrame(coefficients).set_index("seed")
tree_summary = pd.DataFrame(tree_summary).set_index("seed")

print("Test-set performance:")
display(metrics.round(3))

print("Linear-model coefficients by seed:")
display(coefficients.round(4))

print("Coefficient standard deviations across seeds:")
display(coefficients.std().rename("standard_deviation").to_frame().round(4))

print("Tree size and root split by seed:")
display(tree_summary.round(3))

print("Predictions for test samples:")
display(prediction.head(10))

c:\Users\phaml\OneDrive\Documents\DS3001\A01-data-wrangling\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\phaml\OneDrive\Documents\DS3001\A01-data-wrangling\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as s

Test-set performance:


c:\Users\phaml\OneDrive\Documents\DS3001\A01-data-wrangling\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,seed,model,MAE,RMSE,R2
0,42,tree,0.233,0.483,-0.149
1,42,linear,0.217,0.465,-0.067
2,65,tree,0.233,0.483,-0.149
3,65,linear,0.233,0.483,-0.149
4,28,tree,0.217,0.465,-0.067
5,28,linear,0.233,0.483,-0.149
6,2,tree,0.233,0.483,-0.149
7,2,linear,0.250,0.500,-0.231


Linear-model coefficients by seed:


,intercept
seed,
42,0.0363
65,0.0577
28,0.0473
2,0.0850


... (2 outputs left out)

Tree size and root split by seed:


,depth,leaves,root_feature,root_threshold
seed,,,,
42,8,29,time,73.5
65,8,28,time,73.5
28,9,33,time,67.0
2,8,26,time,73.5


Predictions for test samples:


,actual,tree_42,linear_42,tree_65,linear_65,tree_28,linear_28,tree_2,linear_2
217,1,1,0,0,0,0,0,1,0
213,1,0,0,0,0,0,0,1,0
38,0,1,1,1,1,1,1,1,1
259,0,0,0,0,0,0,0,0,0
267,0,0,0,0,0,0,0,0,0
108,0,0,0,0,0,0,0,0,0
67,1,1,1,1,1,1,1,1,1
296,0,0,0,0,0,0,0,0,0
147,0,0,0,0,0,0,0,0,0
211,0,0,0,0,0,0,0,0,0


The trees we get are similar, having around the same depth and amount of leaves. There is only one outlier, which is a depth of 9 and has 33 leaves, but that is only off by one level. The coefficients are very small, with a standard deviation of .0208.


4. Typically, you would see the trees changing what appears to be a non-trivial amount, while the linear model coefficients don't vary nearly as much. Often, the changes appear substantial. 
But are they? Instead of focusing on the tree or model coefficients, do three things:
    
    a. Make scatterplots of the predicted values on the test set from question 2 against the predicted values for the alternative models from part 3, separately for your trees and linear models. Do they appear reasonably similar?
    
    b. Compute the correlation between your model in part 2 and your alternative models in part 3, separately for your trees and linear models. Are they highly correlated or not?
    
    c. Run a simple linear regression of the predicted values on the test set from the alternative models on the predicted values from question 2, separately for your trees and linear models. Is the intercept close to zero? Is the slope close to 1? Is the $R^2$ close to 1?


5. Do linear models appear to have similar coefficients and predictions across train/test splits? Do trees?


6. True or false for this dataset, and explain: "Even if the models end up having a substantially different appearance, the predictions they generate are often very similar."

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]